In [0]:
import os
import sys
import yaml
from databricks.sdk import WorkspaceClient

# t29_role_access: what this login may do in this company's dev, test and prod = the access matrix.
# Run by each person (Developer, Team Lead, Manager, Management) from their own dev Git folder, in both repos.
# Read-only, except one deliberate "write must fail" test per catalog (a table it manages to create is dropped at once).
# Prints no emails, tokens or secret values. Same file in both repos; the company comes from config/company.yml.
# The expected grants come from the access matrix in src/lh/access.py (the same one 01_grant_access applies).
# FAIL = access differs from the matrix; WARN = not built yet (Gold before the first run of that environment,
# your own dev job before you first deploy it).
root = os.path.dirname(os.getcwd())
sys.path.insert(0, os.path.join(root, "src"))
from lh import access  # noqa: E402  the access matrix (also used by setup_once/notebook/01_grant_access)

code = yaml.safe_load(open(os.path.join(root, "config", "company.yml")))["code"]
w = WorkspaceClient()
results = []


def check(level, name, ok, detail=""):
    status = "PASS" if ok else level
    results.append(status)
    print(f"{status:4} {name}", detail)


def q(sql):
    return spark.sql(sql).collect()


def why(e):
    m = str(e)
    for k in ("PERMISSION_DENIED", "INSUFFICIENT_PERMISSIONS", "UNAUTHORIZED", "Forbidden", "does not exist"):
        if k.lower() in m.lower():
            return k
    return f"{type(e).__name__}: {m.splitlines()[0][:80]}"


# 1. who am I: exactly one role group; only Management is workspace admin
ROLES = {"lh_developer": "developer", "lh_teamlead": "teamlead", "lh_manager": "manager", "lh_management": "management"}
member = {g: q(f"SELECT is_account_group_member('{g}')")[0][0] for g in list(ROLES) + ["lh_bizmanager"]}
mine = [r for g, r in ROLES.items() if member[g]]
check("FAIL", "exactly one role group", len(mine) == 1, mine)
assert len(mine) == 1, "this login must be in exactly one of lh_developer, lh_teamlead, lh_manager, lh_management"
role, group = mine[0], f"lh_{mine[0]}"
print(f"     company = {code}, role = {role}")
check("FAIL", "not in lh_bizmanager", not member["lh_bizmanager"])
admin = q("SELECT is_member('admins')")[0][0]
check("FAIL", f"workspace admin = {role == 'management'}", admin == (role == "management"))

# 2. catalogs: grants per the matrix, plus live tests (read Gold, raw files, write must fail)
R = {"USE CATALOG", "SELECT"}


def expect(env):
    """(must have, must not have) on the catalog for this role, read from the access matrix (src/lh/access.py)."""
    granted = {p for n, scope, ps in access.matrix(code, env) if n == group and scope == "catalog" for p in ps}
    watched = {"MODIFY", "READ VOLUME", "MANAGE"}
    return R | (granted & watched), watched - granted


for env in ("dev", "test", "prod"):
    c = f"{code}_{env}"
    need, deny = expect(env)
    try:
        privs = {r["ActionType"].replace("_", " ").upper()
                 for r in q(f"SHOW GRANTS ON CATALOG {c}") if r["Principal"] == group}
    except Exception as e:
        check("FAIL", f"{c}: catalog visible", False, f"({why(e)})")
        continue
    check("FAIL", f"{c}: has {sorted(need)}", need <= privs, sorted(privs))
    bad = privs & (deny | {"ALL PRIVILEGES"})
    check("FAIL", f"{c}: has none of {sorted(deny)}", not bad, sorted(bad) or "")

    tabs = [r[0] for r in q(f"SELECT table_name FROM {c}.information_schema.tables "
                            "WHERE table_schema = 'gold' ORDER BY table_name")]
    if not tabs:
        check("FAIL" if env == "dev" else "WARN", f"{c}: Gold readable", False,
              f"(no Gold tables yet: built by the first {env} run)")
    else:
        try:
            n = q(f"SELECT count(*) FROM {c}.gold.`{tabs[0]}`")[0][0]
            check("FAIL", f"{c}: Gold readable", True, f"({tabs[0]}: {n} rows)")
        except Exception as e:
            check("FAIL", f"{c}: Gold readable", False, f"({why(e)})")

    want_files = "READ VOLUME" in need
    try:
        dbutils.fs.ls(f"/Volumes/{c}/bronze/landing")
        got, reason = True, ""
    except Exception as e:
        got, reason = False, f"({why(e)})"
    check("FAIL", f"{c}: raw files {'readable' if want_files else 'blocked'}", got == want_files, reason)

    if "MODIFY" in deny:
        t = f"{c}.gold.zz_t29_write_{role}"
        try:
            q(f"CREATE TABLE {t} (x INT)")
            q(f"DROP TABLE IF EXISTS {t}")
            check("FAIL", f"{c}: write blocked", False, "(a table could be created; dropped again)")
        except Exception as e:
            check("FAIL", f"{c}: write blocked", True, f"({why(e)})")

# 3. jobs: test/prod jobs visible; your own dev job (not for Management)
names = {j.settings.name for j in w.jobs.list() if j.settings and (j.settings.name or "").startswith(f"lh_{code}_")}
for env in ("test", "prod"):
    n = f"lh_{code}_{env}_daily"
    check("FAIL", f"job {n} visible", n in names)
if role != "management":
    n = f"lh_{code}_dev_daily_{role}"
    check("WARN", f"your dev job {n} exists", n in names,
          "" if n in names else "(made the first time you run jobs/01_deploy_jobs in dev)")


# 4. secrets: your own dev mail key readable (needed to deploy your dev job); test/prod scopes closed
def readable(scope, key):
    try:
        return len(dbutils.secrets.get(scope, key)) > 0
    except Exception:
        return False


if role != "management":
    check("FAIL", f"lh_dev/mail_{role} readable (deploy your dev job)", readable("lh_dev", f"mail_{role}"))
    for env in ("test", "prod"):
        check("FAIL", f"lh_{env} secrets not readable", not readable(f"lh_{env}", "mail_developer"))

# 4b. vault (keyed token masks, ADR 0010): no working role reads the token key or calls the token function.
# Counting rows never fetches the key. Management owns the vault catalog (it created it) and sees real values anyway.
if role != "management":
    for env in ("test", "prod"):
        v = f"{access.VAULT}.{code}_{env}"
        for what, sql in (("token key", f"SELECT count(*) FROM {v}.mask_key"),
                          ("token function", f"SELECT {v}.mask_token('X1')")):
            try:
                q(sql)
                check("FAIL", f"{v}: {what} blocked", False, "(it answered: access must be removed)")
            except Exception as e:
                check("FAIL", f"{v}: {what} blocked", True, f"({why(e)})")


# 5. Git: exactly one credential, this role's own GitHub account
creds = list(w.git_credentials.list())
users = [c.git_username for c in creds]
check("FAIL", "one Git credential", len(creds) == 1, f"({len(creds)})")
check("FAIL", f"Git credential = lh-{role}", users == [f"lh-{role}"], users)

print(f"\n{code} {role}: {results.count('PASS')}/{len(results)} PASS, {results.count('WARN')} WARN, {results.count('FAIL')} FAIL")
assert "FAIL" not in results, "t29_role_access failed"